# ResNeXt — Aggregated Residual Transformations for Deep Neural Networks

**Citation:** Xie, S., Girshick, R., Dollár, P., Tu, Z., & He, K. (2017). Aggregated residual transformations for deep neural networks. *CVPR 2017*. arXiv:1611.05431.

**Registered names:** `resnext50_32x4d`, `resnext101_32x4d`, `resnext101_32x8d`

**Summary:** ResNeXt introduces *cardinality* — the number of parallel transformation groups — as a new dimension of scaling beyond depth and width. A 32×4d ResNeXt-50 matches ResNet-50 accuracy at fewer FLOPs, and matches ResNet-101 at ResNet-50's complexity.

---

| Model | Layers | C × D | Params | Top-1 |
|-------|--------|-------|--------|-------|
| ResNeXt-50 (32×4d) | 50 | 32 × 4 | 25.0M | 77.6% |
| ResNeXt-101 (32×4d) | 101 | 32 × 4 | 44.2M | 78.8% |
| ResNeXt-101 (32×8d) | 101 | 32 × 8 | 88.8M | 79.3% |

## Architecture Diagram

```
ResNet Bottleneck vs ResNeXt Bottleneck (32×4d)

ResNet:                    ResNeXt (32×4d):
─────────────────          ──────────────────────────
in [B, C_in, H, W]         in [B, C_in, H, W]
      │                          │
 Conv 1×1 (→planes)         Conv 1×1 (→128)      ← width = C*D = 32*4 = 128
      │                          │
 Conv 3×3 (→planes)         GroupConv 3×3        ← groups=32 (cardinality)
      │                     (128→128, g=32)
 Conv 1×1 (→planes*4)            │
      │                     Conv 1×1 (→planes*4)
 + residual                      │
      │                    + residual
 ReLU                       ReLU

Insight: 32 groups of width-4 convolutions = 1 grouped conv.
Same FLOPs as ResNet but captures 32 independent transformation paths.

Full network (same as ResNet):
  Stem → Stage0(256ch) → Stage1(512ch) → Stage2(1024ch) → Stage3(2048ch)
  → GlobalAvgPool → pooled [B, 2048]
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor('#0f1117')

def draw_bottleneck(ax, title, layers, colors, x_start=1.5):
    ax.set_facecolor('#0f1117')
    ax.set_xlim(0, 6); ax.set_ylim(0, 10); ax.axis('off')
    ax.text(3, 9.5, title, ha='center', color='white', fontsize=12, fontweight='bold')

    y = 8.5
    for label, color in zip(layers, colors):
        ax.add_patch(FancyBboxPatch((1.0, y - 0.4), 4.0, 0.8,
                                     boxstyle='round,pad=0.08', facecolor=color,
                                     edgecolor='white', linewidth=0.8, alpha=0.88))
        ax.text(3.0, y, label, ha='center', va='center', color='white', fontsize=9, fontweight='bold')
        if y > 2.5:
            ax.annotate('', xy=(3.0, y - 0.9), xytext=(3.0, y - 0.4),
                        arrowprops=dict(arrowstyle='->', color='white', lw=1.2))
        y -= 1.3

    # Residual connection
    ax.annotate('', xy=(0.5, y + 0.4), xytext=(0.5, 8.9),
                arrowprops=dict(arrowstyle='->', color='#ff9f43', lw=1.5,
                                connectionstyle='arc3,rad=0.3'))
    ax.text(0.3, 5.5, 'Skip\nConn.', ha='center', color='#ff9f43', fontsize=7)

resnet_layers = [
    'Input [B, C_in]',
    'Conv 1×1\n(C_in → planes)',
    'Conv 3×3\n(planes → planes)',
    'Conv 1×1\n(planes → planes×4)',
    '+ Residual\nReLU',
]
resnet_colors = ['#444455', '#4a9eff', '#6bcb77', '#4a9eff', '#ff9f43']
draw_bottleneck(axes[0], 'ResNet Bottleneck', resnet_layers, resnet_colors)

resnext_layers = [
    'Input [B, C_in]',
    'Conv 1×1\n(C_in → C×D = 128)',
    'GroupConv 3×3\n(128 → 128, groups=32)',
    'Conv 1×1\n(128 → planes×4)',
    '+ Residual\nReLU',
]
resnext_colors = ['#444455', '#4a9eff', '#a29bfe', '#4a9eff', '#ff9f43']
draw_bottleneck(axes[1], 'ResNeXt Bottleneck (32×4d)', resnext_layers, resnext_colors)

# GroupConv annotation
axes[1].text(5.5, 6.5, '32 parallel\ntransformation\npaths', ha='center',
             color='#a29bfe', fontsize=8, style='italic',
             bbox=dict(boxstyle='round', facecolor='#1a1a2e', edgecolor='#a29bfe', alpha=0.8))

plt.suptitle('ResNet vs ResNeXt Bottleneck', color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Innovations

### 1. Cardinality — A New Dimension of Scaling
ResNeXt proposes cardinality $C$ (number of transformation groups) as the third axis of scaling:
- **Depth** (ResNet: more layers)
- **Width** (WideResNet: wider channels)
- **Cardinality** (ResNeXt: more transformation groups)

The bottleneck width $D$ is chosen so total FLOPs ≈ ResNet:
$$\text{mid\_ch} = \left\lfloor \frac{\text{planes}}{64} \cdot D \right\rfloor \times C$$

### 2. Three Equivalent Formulations
The paper shows three mathematically equivalent implementations:
1. **Aggregated transformation**: $C$ independent branches summed before projection
2. **Grouped convolution**: Single `Conv(groups=C)` call (most efficient)
3. **Concatenation shortcut**: Input split into $C$ groups, processed independently, concatenated

We use formulation 2 (grouped convolution) for efficiency:
```python
self.conv2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                       padding=1, groups=cardinality, bias=False)
```

### 3. Scaling Law
Experimental finding: increasing cardinality is more effective than increasing width at the same model complexity. A 32×4d ResNeXt-101 outperforms a 1×64d (standard ResNet-101) with similar parameters.

## Code Walkthrough

Implemented in `hale_core/nn/vision/cnn/resnext.py`.

**ResNeXtBottleneck — grouped conv as aggregated transformation:**
```python
class ResNeXtBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1,
                 cardinality=32, base_width=4, downsample=None):
        # mid_ch = C * D; at ResNet's 'planes=64', C=32, D=4 → mid_ch=128
        mid_ch = int(planes * (base_width / 64.0)) * cardinality
        self.conv1 = nn.Conv2d(in_planes, mid_ch, kernel_size=1, bias=False)
        self.conv2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, stride=stride,
                               padding=1, groups=cardinality, bias=False)  # KEY
        self.conv3 = nn.Conv2d(mid_ch, planes * 4, kernel_size=1, bias=False)
```

The only change vs ResNet's Bottleneck is `groups=cardinality` in `conv2`. Everything else (stage structure, shortcut projection, weight init) is identical.

In [ ]:
import torch
import sys
sys.path.insert(0, '../../src')

from hale_core.nn.vision.registry import build_vision_backbone

for name in ['resnext50_32x4d', 'resnext101_32x4d', 'resnext101_32x8d']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        out = bb(x)
    print(f'{name}:')
    print(f'  pooled:       {tuple(out.pooled_output.shape)}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params:       {bb.num_parameters()/1e6:.1f}M')

## References

| Paper | Venue | Key Contribution |
|-------|-------|------------------|
| Xie et al. (2017). Aggregated Residual Transformations. | CVPR 2017 | ResNeXt — cardinality as scaling dimension |
| He et al. (2016). Deep Residual Learning. | CVPR 2016 | ResNet — skip connections |
| Zagoruyko & Komodakis (2016). Wide Residual Networks. | BMVC 2016 | WideResNet — width scaling |
| Howard et al. (2017). MobileNets. | arXiv 2017 | Depthwise separable = extreme cardinality case |
| Krizhevsky et al. (2012). ImageNet Classification. | NeurIPS 2012 | AlexNet — grouped conv in original paper |

**Full Citation:**
> Xie, S., Girshick, R., Dollár, P., Tu, Z., & He, K. (2017). Aggregated residual transformations for deep neural networks. In *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition* (pp. 1492–1500). arXiv:1611.05431.